<div style="background:linear-gradient(135deg, #1e3c72, #2a5298); border-radius:8px; padding:16px 20px; color:white;">
  <div style="font-size:80%; letter-spacing:2px; opacity:0.8;">LLM STUDIES · WEEK 1</div>
  <div style="font-size:150%; font-weight:700; margin-top:2px;">Remembering LLM Bases</div>
  <div style="font-size:90%; opacity:0.9; margin-top:6px;">Torch → Neural Nets → Tokenizer → Embeddings → Positional Encoding</div>
</div>

**Dataset:** [Tiny Shakespeare](https://www.kaggle.com/datasets/kaushaltiwari/tiny-shakespeare) · **Runs on:** local dev container

<div style="background:#2a5298; border-radius:6px; padding:6px 14px; color:white; font-size:110%; font-weight:600;">
  <a id="index"></a>Contents
</div>

<div style="line-height:1.6; padding:2px 8px;">
<div style="margin-top:6px;"><a href="#sec-1"><b>1. Libraries</b></a></div>
<div style="margin-top:6px;"><a href="#sec-2"><b>2. Data</b></a></div>
<div style="margin-top:6px;"><a href="#sec-3"><b>3. Neural network</b></a></div>
<div style="margin-left:22px;"><a href="#sec-3-1">3.1 Basic neuron</a></div>
<div style="margin-left:22px;"><a href="#sec-3-2">3.2 Applying propagation</a></div>
<div style="margin-left:22px;"><a href="#sec-3-3">3.3 Implementing a neural network with torch.nn</a></div>
<div style="margin-left:22px;"><a href="#sec-3-4">3.4 Training a toy classifier</a></div>
<div style="margin-left:22px;"><a href="#sec-3-5">3.5 Training loop and evaluation</a></div>
<div style="margin-left:22px;"><a href="#sec-3-6">3.6 Save and load weights</a></div>
<div style="margin-top:6px;"><a href="#sec-4"><b>4. Tokenizer and embeddings</b></a></div>
<div style="margin-left:22px;"><a href="#sec-4-1">4.1 Manual implementation</a></div>
<div style="margin-left:22px;"><a href="#sec-4-2">4.2 Linear implementation with torch</a></div>
<div style="margin-left:22px;"><a href="#sec-4-3">4.3 Data loader</a></div>
<div style="margin-left:22px;"><a href="#sec-4-4">4.4 Token embedding with torch</a></div>
<div style="margin-left:22px;"><a href="#sec-4-5">4.5 Positional embedding</a></div>
</div>

<div style="background:#2a5298; border-radius:6px; padding:6px 14px; color:white; font-size:110%; font-weight:600;">
  <a id="sec-1"></a>1. Libraries
</div>

In [ ]:
from dotenv import load_dotenv

import tiktoken
import torch
from torch.autograd import grad

from src.nets import NeuralNetwork
from src.tokenizer import ToyDataset, create_dataloader
from src.config import DATA, MODELS

load_dotenv()
torch.manual_seed(123)


<div style="background:#2a5298; border-radius:6px; padding:6px 14px; color:white; font-size:110%; font-weight:600;">
  <a id="sec-2"></a>2. Data
</div>

In [34]:
# Tokenizer using gt2
full_path = DATA / "tiny-shakespeare.txt"
with open(full_path, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Length of dataset in characters: {len(text)}")

Length of dataset in characters: 1115394


<div style="background:#2a5298; border-radius:6px; padding:6px 14px; color:white; font-size:110%; font-weight:600;">
  <a id="sec-3"></a>3. Neural network
</div>

<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-3-1"></a>3.1 Basic neuron
</div>

In [ ]:
import torch.nn.functional as F

y = torch.tensor([1])  # True Label
x1 = torch.tensor([1.1])  # Input Feature
w = torch.tensor([0.5])  # Weight
b = torch.tensor([0.0])  # Bias

z = w * x1 + b  # net input
a = torch.sigmoid(z)  # activation output

loss = F.binary_cross_entropy(a, y.float())
print(f"Loss: {loss.item()}")


Loss: 0.45549246668815613


<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-3-2"></a>3.2 Applying propagation
</div>

In [ ]:
y = torch.tensor([1])
x1 = torch.tensor([1.1])
w1 = torch.tensor([0.5], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = w1 * x1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y.float())

print(f"Loss: {loss.item()}")

# Compute gradients
grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(f"Gradient of loss w.r.t weight: {grad_L_w1[0].item()}")
print(f"Gradient of loss w.r.t bias: {grad_L_b[0].item()}")

Loss: 0.45549246668815613
Gradient of loss w.r.t weight: -0.402450829744339
Gradient of loss w.r.t bias: -0.3658643960952759


In [4]:
loss.backward()

print(w1.grad)
print(b.grad)

tensor([-0.4025])
tensor([-0.3659])


<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-3-3"></a>3.3 Implementing a neural network with torch.nn
</div>

In [ ]:
torch.manual_seed(1234)

# Crate a model
model = NeuralNetwork(50, 3)
print(model)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total number of trainable parameters: {num_params}\n")

# Showing Weights and Biases of the first layer
first_layer = model.layers[0]
print("Weights of the first layer:\n", first_layer.weight, "\n")
print("Biases of the first layer:\n", first_layer.bias)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)
Total number of trainable parameters: 2213

Weights of the first layer:
 Parameter containing:
tensor([[-0.1332, -0.0277, -0.0679,  ..., -0.1229,  0.1086,  0.0872],
        [ 0.0715,  0.1128,  0.0520,  ...,  0.0622,  0.0734,  0.0088],
        [ 0.0410,  0.0629, -0.0165,  ...,  0.1351,  0.0509, -0.0523],
        ...,
        [-0.0326, -0.1390,  0.0027,  ...,  0.0155, -0.0064,  0.0576],
        [-0.0843, -0.0869,  0.0910,  ...,  0.1006, -0.0516,  0.0199],
        [ 0.0013, -0.0454,  0.0163,  ..., -0.0539, -0.0676,  0.1381]],
       requires_grad=True) 

Biases of the first layer:
 Parameter containing:
tensor([ 0.0461,  0.0244, -0.0678, -0.0378,  0.1271,  0.0593,  0.0638, -0.0139,
        -0.1346,  0.0286,  0.1091,  0.1008,  0.0430, -0.

In [ ]:
model = NeuralNetwork(50, 3)
print(model.layers[0].weight)

# Testing
X = torch.randn(5, 50)  # 5 samples, each with 50 features
output = model(X)
print("Output of the model:\n", output, "\n")

with torch.no_grad():
    output = model(X)
    print("Output of the model (no gradient tracking):\n", output, "\n")

with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

Parameter containing:
tensor([[ 0.0514,  0.0681,  0.1271,  ..., -0.1234, -0.0547,  0.0098],
        [-0.0399,  0.0550, -0.0539,  ..., -0.0635, -0.1025,  0.0496],
        [-0.0011, -0.1388,  0.0867,  ..., -0.1021,  0.0794,  0.0750],
        ...,
        [ 0.1170, -0.0153, -0.1215,  ...,  0.0615, -0.0244,  0.0773],
        [-0.0615,  0.0766,  0.1048,  ..., -0.1046,  0.0607, -0.0411],
        [ 0.1092, -0.0138, -0.1407,  ..., -0.0395,  0.1375, -0.0586]],
       requires_grad=True)
Output of the model:
 tensor([[-0.1237, -0.1152, -0.1264],
        [-0.0704, -0.1625, -0.1347],
        [-0.1485, -0.1345, -0.0574],
        [-0.0460, -0.2495, -0.2094],
        [-0.0567, -0.2224, -0.1349]], grad_fn=<AddmmBackward0>) 

Output of the model (no gradient tracking):
 tensor([[-0.1237, -0.1152, -0.1264],
        [-0.0704, -0.1625, -0.1347],
        [-0.1485, -0.1345, -0.0574],
        [-0.0460, -0.2495, -0.2094],
        [-0.0567, -0.2224, -0.1349]]) 

tensor([[0.3327, 0.3355, 0.3318],
        [0.350

<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-3-4"></a>3.4 Training a toy classifier
</div>

In [ ]:
# Creating extra data
X_train = torch.tensor([[-1.2, 3.1], [-0.9, 2.9], [-0.5, 2.6], [2.3, -1.1], [2.7, -1.5]])

X_test = torch.tensor(
    [
        [-0.8, 2.8],
        [2.6, -1.6],
    ]
)

y_train = torch.tensor([0, 0, 0, 1, 1])
y_test = torch.tensor([0, 1])

# Creating ToyDataset instances for training and testing
train_ids = ToyDataset(X_train, y_train)
test_ids = ToyDataset(X_test, y_test)

# Create Data Loader
train_loader = torch.utils.data.DataLoader(train_ids, batch_size=2, shuffle=True, drop_last=True)
test_loader = torch.utils.data.DataLoader(test_ids, batch_size=2, shuffle=False, drop_last=True)

for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
    print(f"Batch {batch_idx + 1}:")
    print("X_batch:\n", X_batch)
    print("y_batch:\n", y_batch)
    print()

Batch 1:
X_batch:
 tensor([[ 2.7000, -1.5000],
        [-0.9000,  2.9000]])
y_batch:
 tensor([1, 0])

Batch 2:
X_batch:
 tensor([[ 2.3000, -1.1000],
        [-0.5000,  2.6000]])
y_batch:
 tensor([1, 0])



<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-3-5"></a>3.5 Training loop and evaluation
</div>

In [ ]:
# Doing a Typical Training Loop

model = NeuralNetwork(X_train.shape[1], len(torch.unique(y_train)))
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
num_epochs = 3

for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
        logtis = model(X_batch)  # Trained
        loss = F.cross_entropy(logtis, y_batch)  # Compute loss
        optimizer.zero_grad()  # Zero the gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        ### LOGGING
        print(
            f"Epoch: {epoch + 1:03d}/{num_epochs:03d}"
            f" | Batch {batch_idx + 1:03d}/{len(train_loader):03d}"
            f" | Loss: {loss:.2f}"
        )

    model.eval()
    # Optional model evaluation


Epoch: 001/003 | Batch 001/002 | Loss: 0.53
Epoch: 001/003 | Batch 002/002 | Loss: 0.67
Epoch: 002/003 | Batch 001/002 | Loss: 0.66
Epoch: 002/003 | Batch 002/002 | Loss: 0.66
Epoch: 003/003 | Batch 001/002 | Loss: 0.65
Epoch: 003/003 | Batch 002/002 | Loss: 0.64


In [22]:
# Check on test without gradient tracking
model.eval()
with torch.no_grad():
    outputs = model(X_train)
print(f"Model outputs on test data:\n{outputs}", "\n")

torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

predictions = torch.argmax(probas, dim=1)
print(predictions)


Model outputs on test data:
tensor([[ 0.0929, -0.4105],
        [ 0.0817, -0.4074],
        [ 0.0673, -0.4015],
        [ 0.1049, -0.0936],
        [ 0.1323, -0.0614]]) 

tensor([[0.6233, 0.3767],
        [0.6199, 0.3801],
        [0.6151, 0.3849],
        [0.5495, 0.4505],
        [0.5483, 0.4517]])
tensor([0, 0, 0, 0, 0])


In [ ]:
def compute_accuracy(model, dataloader) -> float:

    model.eval()  # Set the model to evaluation mode
    correct = 0
    total_examples = 0

    for _, (features, labels) in enumerate(dataloader):
        # Deactivate gradient tracking for evaluation
        with torch.no_grad():
            logits = model(features)

            # Update counts
            predictions = torch.argmax(logits, dim=1)
            compare = labels == predictions
            correct += torch.sum(compare)
            total_examples += len(compare)

    return (correct / total_examples).item()


# Compute Accuracy on the training | Testing set
print(f"accuracy in training {compute_accuracy(model, train_loader)}")
print(f"accuracy in testing {compute_accuracy(model, test_loader)}")

# Saving Model Weights


accuracy in training 0.5
accuracy in testing 0.5


<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-3-6"></a>3.6 Save and load weights
</div>

In [ ]:
# Save and load model weights
torch.save(model.state_dict(), MODELS / "model-week1.pth")

# Load again the model
model2 = NeuralNetwork(X_train.shape[1], len(torch.unique(y_train)))
model2.load_state_dict(torch.load(MODELS / "model-week1.pth", weights_only=True))

<All keys matched successfully>

<div style="background:#2a5298; border-radius:6px; padding:6px 14px; color:white; font-size:110%; font-weight:600;">
  <a id="sec-4"></a>4. Tokenizer and embeddings
</div>

A toy word-level tokenizer: a dictionary that maps each word to an id, with `<unk>` for unknown words.

In [ ]:
# To make it simple I will create a Toy Dictioary
toy_text = "I love my kids with all my hear"
splitted_text = toy_text.split(" ")
tokenizer = {word: idx for idx, word in enumerate(set(splitted_text))}
tokenizer["<unk>"] = len(tokenizer)
print(f"My tokenizer size `{len(tokenizer)}` {tokenizer}")

My tokenizer size `8` {'I': 0, 'love': 1, 'hear': 2, 'kids': 3, 'my': 4, 'all': 5, 'with': 6, '<unk>': 7}


<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-4-1"></a>4.1 Manual implementation
</div>

Building (mimicking) a vector embedding by hand.

An embedding maps each token id to a dense vector of real numbers.
It is stored as a matrix W of shape (N, M), where row i is the vector of token i:

- N = vocabulary size of the tokenizer (50257 for GPT-2 BPE), regardless of
how many of those tokens actually appear in our text.

- M = embedding dimension (e.g. 768 in GPT-2 small). It is a hyperparameter
we choose before training.

W starts with random values and is learned together with the rest of the model.
If we trained our own tokenizer on Tiny Shakespeare, N would be that tokenizer's
vocabulary size. A word-level vocabulary is impractical: any unseen word would
be out of vocabulary, which is why subword tokenizers like BPE are used.

In [ ]:
# Embdding Matrix
dim = 3
ids = torch.tensor([tokenizer[w] for w in splitted_text])
W = torch.arange(len(tokenizer) * dim, dtype=torch.float32).reshape(len(tokenizer), 3)
print(f"Embedding Matrix:\n{W}\n")

# Create One hot encoding
onehot = []
for idx in ids.tolist():
    row = [0 * len(tokenizer)] * len(tokenizer)
    row[idx] = 1
    onehot.append(row)

print(f"One hot encoding:\n{torch.tensor(onehot)}\n")

# Now create the embedding layer using the embedding matrix W
torch.tensor(onehot).float() @ W


Embedding Matrix:
tensor([[ 0.,  1.,  2.],
        [ 3.,  4.,  5.],
        [ 6.,  7.,  8.],
        [ 9., 10., 11.],
        [12., 13., 14.],
        [15., 16., 17.],
        [18., 19., 20.],
        [21., 22., 23.]])

One hot encoding:
tensor([[1, 0, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0, 0]])



tensor([[ 0.,  1.,  2.],
        [ 3.,  4.,  5.],
        [12., 13., 14.],
        [ 9., 10., 11.],
        [18., 19., 20.],
        [15., 16., 17.],
        [12., 13., 14.],
        [ 6.,  7.,  8.]])

<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-4-2"></a>4.2 Linear implementation with torch
</div>

The same lookup, now as a `Linear` layer applied to one-hot vectors.

In [31]:
# Transform the ids into one-hot encoding using PyTorch's built-in function
onehot = torch.nn.functional.one_hot(ids, num_classes=len(tokenizer)).float()

# Now create the embedding layer using the embedding matrix W
linear = torch.nn.Linear(len(tokenizer), dim, bias=False)
linear.weight = torch.nn.Parameter(W.T)
linear(onehot)

tensor([[ 0.,  1.,  2.],
        [ 3.,  4.,  5.],
        [12., 13., 14.],
        [ 9., 10., 11.],
        [18., 19., 20.],
        [15., 16., 17.],
        [12., 13., 14.],
        [ 6.,  7.,  8.]], grad_fn=<MmBackward0>)

<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-4-3"></a>4.3 Data loader
</div>

This creates the dataset for training and testing. The challenge here is choosing `batch_size`, `max_length` and `stride`.

In [ ]:
# MaxLength is the maximum length of the sequence we want to feed into the model.
max_length = 128

# Generate Data
dataloader = create_dataloader(
    data=text,
    batch_size=64,
    max_length=max_length,
    stride=max_length,  # This avoid Overlapping Sequences
)

data_iter = iter(dataloader)
inputs, target = next(data_iter)
print("Inputs:\n", inputs)
print("target:\n", target)

Inputs:
 tensor([[   11, 15967,    11,  ...,   287,  5897,    11],
        [   77,  6243,  1275,  ...,   198,    40,   783],
        [  340,    11,   290,  ...,  1182,     0,   198],
        ...,
        [15876,    26,   290,  ...,   307,  1498,   284],
        [ 8001, 14210,  1654,  ...,    32,  1014,     0],
        [ 1719,   656,  3872,  ...,  3595,   582,    11]])
target:
 tensor([[15967,    11,   307,  ...,  5897,    11, 44642],
        [ 6243,  1275,   270,  ...,    40,   783,  2221],
        [   11,   290,   750,  ...,     0,   198,   198],
        ...,
        [   26,   290,   523,  ...,  1498,   284,  8791],
        [14210,  1654,   286,  ...,  1014,     0,  1309],
        [  656,  3872,    11,  ...,   582,    11,   616]])


<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-4-4"></a>4.4 Token embedding with torch
</div>

In [36]:
# Embeddings
output_dim = 256
vocab_size = tiktoken.get_encoding("gpt2").n_vocab  # 50257

# Embedding matrix W of shape (vocab_size, output_dim), randomly initialized.
# It is learned during training together with the rest of the model.
token_embedding_layer = torch.nn.Embedding(num_embeddings=vocab_size, embedding_dim=output_dim)

# Lookup: each token id in `inputs` selects its row of W
token_embedding = token_embedding_layer(inputs)
batch_size, seq_len, emb_dim = token_embedding.shape
print(f"Embedding output [batch_size, seq_len, emb_dim]: {token_embedding.shape}")
print(f"{batch_size} sequences, {seq_len} tokens each, {emb_dim} numbers per token")


Embedding output [batch_size, seq_len, emb_dim]: torch.Size([64, 128, 256])
64 sequences, 128 tokens each, 256 numbers per token


<div style="border-left:3px solid #4f83e0; padding:1px 10px; font-weight:600;">
  <a id="sec-4-5"></a>4.5 Positional embedding
</div>

From this point adding the positional encoding into the vector embedding to shift | move the vector for the transformer to identified the best vector position
we just sum them into the vector space , this would affect all the batches

In [ ]:
# Generate Positional Econding
post_embedding_layer = torch.nn.Embedding(num_embeddings=max_length, embedding_dim=output_dim)
print(f"Dim: {post_embedding_layer}", "\n")

input_embedding = token_embedding + post_embedding_layer(torch.arange(seq_len))
print(f"Input Embedding:\n{input_embedding}", "\n")

Dim: Embedding(128, 256) 

Input Embedding:
tensor([[[ 0.4694, -1.6320, -0.2709,  ...,  0.8581,  1.0578,  0.5506],
         [ 0.2702, -0.4971,  0.2619,  ...,  0.6610,  0.6027, -0.4364],
         [-2.1348, -1.9221,  0.5239,  ...,  0.1068,  0.8061, -0.9046],
         ...,
         [ 0.6609,  0.2691, -0.9726,  ..., -0.9408, -1.3474,  1.8791],
         [ 1.4877, -1.2918,  0.0621,  ..., -0.4917, -0.1761, -1.6591],
         [ 0.4364, -1.0930,  0.9721,  ..., -1.5924,  1.0923,  0.4269]],

        [[ 0.5229, -0.5372,  1.6073,  ...,  1.9726,  0.4796,  1.7309],
         [ 0.0774, -2.2603,  1.4476,  ..., -1.5105,  0.6517, -0.0900],
         [-2.4619, -0.6619, -1.7345,  ...,  1.2149, -0.0297, -1.9798],
         ...,
         [ 1.8562, -1.1432, -0.1695,  ..., -0.7692, -0.0972,  1.3120],
         [-0.2993, -1.8760, -2.2866,  ...,  0.5334,  2.3713, -0.1316],
         [-0.0591,  0.4070,  1.4472,  ..., -1.6189,  0.7885,  1.0540]],

        [[ 0.8773, -0.3089, -2.2383,  ...,  2.3252, -0.3874,  0.6993],
 